In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

from category_encoders import TargetEncoder

In [ ]:
df = pd.read_csv("/content/used_cars.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['milage'].unique()

In [ ]:
df1 = df.copy()

In [ ]:
df1["milage"] = (
    df["milage"]
    .str.replace(",", "", regex=False)
    .str.replace(" mi.", "", regex=False)
    .astype(int)
)

In [ ]:
df1["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(int)
)

In [ ]:
df1.rename(columns={"milage": "mileage"}, inplace=True)

In [ ]:
df1["clean_title"] = df1["clean_title"].fillna("No")

In [ ]:
categorical_columns = ["fuel_type", "accident"]

for col in categorical_columns:
    df1[col] = df1[col].fillna("Unknown")

In [ ]:
df1["accident"] = (
    df1["accident"]
    .str.contains("accident", case=False, na=False)
    .astype(int)
)

In [ ]:
CURRENT_YEAR = 2026

df1["vehicle_age"] = CURRENT_YEAR - df1["model_year"]

In [ ]:
df1.info()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df1["price"], bins=50)

plt.xlabel("Price")
plt.ylabel("Number of Cars")
plt.title("Used Car Price Distribution")

plt.show()

In [ ]:
plt.boxplot(df1["price"])

In [ ]:
df1=df1[df1['price']<100000]

In [ ]:
df1["accident"].value_counts()

In [ ]:
df1['ext_col'].value_counts()

In [ ]:
ext_col_stats = df1.ext_col.value_counts(ascending=False)
ext_col_less_than_10 = ext_col_stats[ext_col_stats <=10]
df1.ext_col=df1.ext_col.apply(lambda x: 'other' if x in ext_col_less_than_10 else x)
df1.ext_col.nunique()

In [ ]:
df1['int_col'].value_counts()

In [ ]:
int_col_stats = df1.int_col.value_counts(ascending=False)
int_col_less_than_10 = int_col_stats[int_col_stats <=10]
df1.int_col=df1.int_col.apply(lambda x: 'other' if x in int_col_less_than_10 else x)
df1.int_col.nunique()

In [ ]:
df1['brand'].value_counts()

In [ ]:
brand_stats = df1.brand.value_counts(ascending=False)
brand_less_than_10 = brand_stats[brand_stats <=10]
df1.brand=df1.brand.apply(lambda x: 'other' if x in brand_less_than_10 else x)
df1.brand.nunique()

In [ ]:
df1['model'].value_counts()

In [ ]:
model_stats = df1.model.value_counts(ascending=False)
model_less_than_10 = model_stats[model_stats <=10]
df1.model=df1.model.apply(lambda x: 'other' if x in model_less_than_10 else x)
df1.model.nunique()

In [ ]:
df1["mileage_per_year"] = df1["mileage"] / df1["vehicle_age"].replace(0, 1)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(df1["mileage"], df1["price"], alpha=.5)

plt.xlabel("Mileage")
plt.ylabel("Price")
plt.title("Mileage vs Used Car Price")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(df1["vehicle_age"], df1["price"], alpha=0.5)

plt.xlabel("Vehicle Age")
plt.ylabel("Price")
plt.title("Vehicle Age vs Used Car Price")

plt.show()

In [ ]:
brand_prices = (df1.groupby("brand")["price"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False))

brand_prices.head(15)

In [ ]:
corr = df1.select_dtypes("number").corr()
corr.style.background_gradient(axis=None)

In [ ]:
%pip install category_encoders

In [ ]:
X = df1.drop("price", axis=1)
y = df1["price"]

In [ ]:
categorical_features = ["brand", "model", "fuel_type", "engine", "transmission", "ext_col", "int_col", "clean_title"]

numerical_features = ["model_year", "mileage", "vehicle_age", "mileage_per_year", "accident"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split ( X, y, test_size=0.20, random_state=42)

In [ ]:
encoder = TargetEncoder(cols=categorical_features,smoothing=10)

In [ ]:
X_train_encoded = encoder.fit_transform( X_train, y_train)

X_test_encoded = encoder.transform(X_test)

In [ ]:
rf_model = RandomForestRegressor( n_estimators=300, max_depth=None, min_samples_split=2, random_state=42, n_jobs=-1 )

In [ ]:
rf_model.fit( X_train_encoded, y_train)

In [ ]:
rf_predictions = rf_model.predict(X_test_encoded)

In [ ]:
rf_mae = mean_absolute_error( y_test, rf_predictions)
print("Random Forest MAE:", rf_mae)

In [ ]:
rf_mpe = mean_absolute_percentage_error( y_test, rf_predictions)
print("Random Forest MPE:", rf_mpe)

In [ ]:
rf_r2 = r2_score( y_test, rf_predictions)
print("Random Forest R2 Score:", rf_r2)

In [ ]:
gb_model = GradientBoostingRegressor( n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)

In [ ]:
gb_model.fit(
    X_train_encoded,
    y_train
)

In [ ]:
gb_predictions = gb_model.predict(X_test_encoded)

In [ ]:
gb_mae = mean_absolute_error(y_test, gb_predictions)

gb_mpe = mean_absolute_percentage_error(y_test, gb_predictions)

gb_r2 = r2_score(y_test, gb_predictions)

print("Gradient Boosting Results")
print("-------------------------")
print(f"MAE: ${gb_mae:,.2f}")
print(f"MPE: {gb_mpe:.2%}")
print(f"R² Score: {gb_r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        rf_mae,
        gb_mae
    ],
    "R2": [
        rf_r2,
        gb_r2
    ]
})

results.sort_values("MAE")

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, rf_predictions, alpha=0.5, label="Random Forest")
plt.scatter(y_test, gb_predictions, alpha=0.5, label="Gradient Boosting")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Used Car Prices")

plt.show()

In [ ]:
feature_importance = pd.DataFrame({"Feature": X_train_encoded.columns,"Importance": rf_model.feature_importances_})

feature_importance = feature_importance.sort_values("Importance", ascending=False)

feature_importance.head()

In [ ]:
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))

plt.barh(top_features["Feature"], top_features["Importance"])

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Features Influencing Used Car Prices")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
def predict_car_price(car_data):

    car_df = pd.DataFrame([car_data])

    for col in categorical_features:
        car_df[col] = car_df[col].fillna("Unknown")

    encoded = encoder.transform(car_df)

    prediction = rf_model.predict(encoded)

    return prediction[0]

In [ ]:
example_car = {
    "brand": "Toyota",
    "model": "Camry XSE",
    "model_year": 2020,
    "mileage": 35000,
    "fuel_type": "Gasoline",
    "engine": "2.5L I4",
    "transmission": "Automatic",
    "ext_col": "White",
    "int_col": "Black",
    "accident": 0,
    "clean_title": "Yes",
    "vehicle_age": 5,
    "mileage_per_year": 20000,

}

In [ ]:
predicted_price = predict_car_price(example_car)

print(f"Estimated Vehicle Price: ${predicted_price:,.2f}")

In [ ]:
import joblib

joblib.dump(rf_model, "used_car_price_model.pkl")

joblib.dump(encoder, "target_encoder.pkl")

In [ ]:
from google.colab import files

files.download("used_car_price_model.pkl")
files.download("target_encoder.pkl")